# Proj30 — MLflow + GPU Training Setup


## What this notebook does
1. Generates S3-compatible credentials for the Chameleon object store
2. Creates the MLflow artifact bucket
3. Launches an always-on MLflow server VM (KVM@TACC)
4. Launches a GPU training node (KVM@TACC — works with any GPU flavor)
5. Installs Docker + NVIDIA toolkit on the GPU node
6. Clones the training repo and builds the Docker image
7. Writes the credentials file on the MLflow VM
8. Sets up a weekly cron job for automated retraining
9. Runs a smoke test to verify the full pipeline end to end

## Prerequisites
Before running this notebook you need:
- A Chameleon Cloud account with access to project **CHI-251409**
- Two active leases (create these at https://chi.tacc.chameleoncloud.org → Reservations → Leases):
  - **MLflow VM lease** on KVM@TACC (any m1.medium or larger flavor)
  - **GPU training lease** on KVM@TACC (any GPU flavor — g1.v100, g1.h100, etc.)
- SSH key configured in your Chameleon account

---
## About `/home/cc/.mlflow_s3_credentials`

This is a plain text file that lives on the MLflow VM. It stores all environment
variables needed by the automated retraining pipeline. The cron job sources it
before running `retrain_trigger.py`, so no credentials are ever hardcoded in scripts.

The file contains:
```
AWS_ACCESS_KEY_ID          — EC2-compatible key for Chameleon object store (auto-generated)
AWS_SECRET_ACCESS_KEY      — EC2-compatible secret (auto-generated)
MLFLOW_TRACKING_URI        — URL of the MLflow server (set from MLflow VM floating IP)
MLFLOW_S3_ENDPOINT_URL     — Chameleon object store endpoint (fixed URL)
MLFLOW_ARTIFACTS_BUCKET    — bucket where MLflow stores model artifacts
TRAINING_DATA_BUCKET       — bucket where meeting transcripts live (teammate's bucket)
GITHUB_REPO                — repo to clone on the GPU node during automated retraining
OS_PROJECT_NAME            — Chameleon project name (used by chi library)
GPU_NODE_IP                — floating IP of the GPU node (SSH target for cron trigger)
INFERENCE_SERVER_URL       — URL of the inference server /reload endpoint (set by teammate)
```

**This file is never committed to GitHub.** It is written once during setup and
lives only on the MLflow VM.

---
## Cell 1 — Configuration

**Edit the values in the EDIT section before running anything else.**

- `MLFLOW_LEASE_NAME`: the name of your KVM@TACC lease for the MLflow VM
- `GPU_LEASE_NAME`: the name of your KVM@TACC lease for the GPU node
- `INFERENCE_SERVER_URL`: leave blank until your teammate provides the inference server IP

In [ ]:
# ============================================================
# EDIT THESE VALUES
# ============================================================
MLFLOW_LEASE_NAME    = "proj30_sssyu"     # KVM@TACC lease for the MLflow VM
GPU_LEASE_NAME       = "proj30_mltrain"   # KVM@TACC lease for the GPU training node
INFERENCE_SERVER_URL = ""                 # leave blank until teammate provides it

# ============================================================
# DO NOT EDIT BELOW THIS LINE
# ============================================================
RESOURCE_PREFIX         = "proj30"
SERVER_NAME             = f"{RESOURCE_PREFIX}-mlflow-server"
VOLUME_NAME             = f"{RESOURCE_PREFIX}-mlflow-persist"
MLFLOW_ARTIFACTS_BUCKET = f"{RESOURCE_PREFIX}-mlflow-artifacts"
TRAINING_DATA_BUCKET    = "ObjStore_projecttranscriptionmirotalk"
GPU_NODE_NAME           = f"{RESOURCE_PREFIX}-gpu-node"
IMAGE_NAME              = "CC-Ubuntu24.04"
GPU_IMAGE_NAME          = "CC-Ubuntu24.04-CUDA"
VOLUME_SIZE_GB          = 5
GITHUB_REPO             = "miandfetter/ECE-9183-Proj30"
SSH_KEY_PATH            = "~/work/.ssh/id_rsa"  # default Trovi SSH key location

print("Configuration:")
print(f"  MLflow lease:      {MLFLOW_LEASE_NAME}")
print(f"  GPU lease:         {GPU_LEASE_NAME}")
print(f"  MLflow server:     {SERVER_NAME}")
print(f"  GPU node:          {GPU_NODE_NAME}")
print(f"  Artifacts bucket:  {MLFLOW_ARTIFACTS_BUCKET}")
print(f"  Training bucket:   {TRAINING_DATA_BUCKET}")
print(f"  GitHub repo:       {GITHUB_REPO} (public)")

---
## Cell 2 — Imports

In [ ]:
import os
import time
import swiftclient
from chi import context, server, lease, network
import chi

print("Imports OK")

---
## Cell 3 — Select Project + Generate S3 Credentials

Chameleon uses EC2-compatible credentials to access its object store (Swift) via
the S3 API. This cell either loads existing credentials from
`~/work/.mlflow_s3_credentials` or generates new ones automatically.

You do not need to create these manually — they are generated from your
Chameleon OpenStack session.

In [ ]:
context.reset()
context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@TACC")

env_file = os.path.expanduser("~/work/.mlflow_s3_credentials")

if os.path.exists(env_file):
    credentials = {}
    with open(env_file) as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                key, value = line.split("=", 1)
                credentials[key] = value
    AWS_ACCESS_KEY_ID     = credentials["AWS_ACCESS_KEY_ID"]
    AWS_SECRET_ACCESS_KEY = credentials["AWS_SECRET_ACCESS_KEY"]
    print(f"Loaded existing credentials from {env_file}")
else:
    conn        = chi.clients.connection()
    project_id  = conn.current_project_id
    identity_ep = conn.session.get_endpoint(service_type="identity", interface="public")
    url  = f"{identity_ep}/v3/users/{conn.current_user_id}/credentials/OS-EC2"
    resp = conn.session.post(url, json={"tenant_id": project_id})
    resp.raise_for_status()
    ec2 = resp.json()["credential"]
    AWS_ACCESS_KEY_ID     = ec2["access"]
    AWS_SECRET_ACCESS_KEY = ec2["secret"]
    os.makedirs(os.path.dirname(env_file), exist_ok=True)
    with open(env_file, "w") as f:
        f.write(f"AWS_ACCESS_KEY_ID={AWS_ACCESS_KEY_ID}\n")
        f.write(f"AWS_SECRET_ACCESS_KEY={AWS_SECRET_ACCESS_KEY}\n")
    print(f"Generated new credentials and saved to {env_file}")

print(f"AWS_ACCESS_KEY_ID: {AWS_ACCESS_KEY_ID[:8]}... (truncated for security)")

---
## Cell 4 — Create MLflow Artifacts Bucket

Creates the object store bucket where MLflow stores model artifacts (weights,
metrics, etc.). This is separate from the training data bucket which is managed
by another team member.

In [ ]:
context.choose_site(default="CHI@TACC")

os_conn     = chi.clients.connection()
token       = os_conn.authorize()
storage_url = os_conn.object_store.get_endpoint()
swift_conn  = swiftclient.Connection(preauthurl=storage_url, preauthtoken=token, retries=5)

try:
    swift_conn.head_container(MLFLOW_ARTIFACTS_BUCKET)
    print(f"Bucket '{MLFLOW_ARTIFACTS_BUCKET}' already exists — skipping creation")
except swiftclient.exceptions.ClientException:
    swift_conn.put_container(MLFLOW_ARTIFACTS_BUCKET)
    print(f"Created bucket '{MLFLOW_ARTIFACTS_BUCKET}'")

---
## Cell 5 — Launch MLflow VM (KVM@TACC)

Launches the always-on MLflow server VM. This VM runs:
- MLflow tracking server (port 8000)
- PostgreSQL database (backend store for MLflow)
- The weekly cron job that triggers retraining

Uses `idempotent=True` — safe to re-run if the VM already exists.

In [ ]:
context.choose_site(default="KVM@TACC")

l = chi.lease.get_lease(MLFLOW_LEASE_NAME)
print(f"Lease: {l.name} — status: {l.status}")
print(f"Reserved flavors: {[f.name for f in l.get_reserved_flavors()]}")

if not l.get_reserved_flavors():
    raise RuntimeError(f"Lease {MLFLOW_LEASE_NAME} has no reserved flavors. Check the lease in the Chameleon UI.")

s = server.Server(
    SERVER_NAME,
    image_name=IMAGE_NAME,
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)
s.associate_floating_ip()
s.refresh()
s.check_connectivity()

mlflow_ip = s.get_floating_ip()
print(f"\nMLflow VM is up: {mlflow_ip}")
print(f"MLflow UI will be at: http://{mlflow_ip}:8000 (after Docker starts in Cell 8)")

---
## Cell 6 — Security Groups + Persistent Volume for MLflow VM

In [ ]:
# Security groups
for sg in [
    {"name": "allow-ssh",  "port": 22,   "description": "SSH"},
    {"name": "allow-8000", "port": 8000, "description": "MLflow UI"},
    {"name": "allow-8888", "port": 8888, "description": "JupyterLab"},
]:
    secgroup = network.SecurityGroup({"name": sg["name"], "description": sg["description"]})
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])
print("Security groups attached")

# Persistent volume — preserves PostgreSQL data across VM restarts
cinder_client    = chi.clients.cinder()
existing_volumes = [v for v in cinder_client.volumes.list() if v.name == VOLUME_NAME]

if existing_volumes:
    volume = existing_volumes[0]
    print(f"Found existing volume: {volume.name} (status: {volume.status})")
else:
    volume = cinder_client.volumes.create(name=VOLUME_NAME, size=VOLUME_SIZE_GB)
    print(f"Created volume: {volume.name} ({VOLUME_SIZE_GB}GB)")

volume = cinder_client.volumes.get(volume.id)
while volume.status not in ["available", "in-use"]:
    print(f"Volume status: {volume.status}, waiting...")
    time.sleep(5)
    volume = cinder_client.volumes.get(volume.id)

if volume.status == "available":
    s.attach_volume(volume.id)
    print(f"Attached volume {VOLUME_NAME} to MLflow VM")
else:
    print(f"Volume already in use — skipping attach")

---
## Cell 7 — Install Docker + Mount Volume on MLflow VM

In [ ]:
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")

# Format and mount the persistent volume (skips format if already formatted)
volume = cinder_client.volumes.get(volume.id)
device = volume.attachments[0]["device"]
part   = f"{device}1"

s.execute(f"""
set -e
if ! sudo blkid {part} >/dev/null 2>&1; then
    sudo parted -s {device} mklabel gpt
    sudo parted -s {device} mkpart primary ext4 0% 100%
    sudo mkfs.ext4 {part}
fi
sudo mkdir -p /mnt/mlflow_persist
if ! mountpoint -q /mnt/mlflow_persist; then
    sudo mount {part} /mnt/mlflow_persist
fi
sudo chown -R cc:cc /mnt/mlflow_persist
mkdir -p /mnt/mlflow_persist/postgres_data
""")
print("Docker installed and volume mounted at /mnt/mlflow_persist")

---
## Cell 8 — Start MLflow Server via Docker Compose

Starts MLflow + PostgreSQL as Docker containers. The docker-compose file and
credentials are uploaded from this repo.

In [ ]:
# Write .env file locally then upload
os.makedirs("docker", exist_ok=True)
with open("docker/.env", "w") as f:
    f.write(f"AWS_ACCESS_KEY_ID={AWS_ACCESS_KEY_ID}\n")
    f.write(f"AWS_SECRET_ACCESS_KEY={AWS_SECRET_ACCESS_KEY}\n")
    f.write(f"BUCKET_NAME={MLFLOW_ARTIFACTS_BUCKET}\n")

s.execute("mkdir -p ~/docker")
s.upload("docker/docker-compose-mlflow.yaml", "/home/cc/docker/docker-compose-mlflow.yaml")
s.upload("docker/.env", "/home/cc/docker/.env")

s.execute("docker compose -f ~/docker/docker-compose-mlflow.yaml up -d")
time.sleep(10)  # give services a moment to start
s.execute("docker ps")

print(f"\nMLflow UI: http://{mlflow_ip}:8000")

---
## Cell 9 — Launch GPU Training Node (KVM@TACC)

Launches the GPU training node. This works with **any GPU flavor** available
on KVM@TACC — the flavor is read automatically from your lease.

Common GPU flavors on KVM@TACC:
- `g1.v100` — NVIDIA V100 (16GB)
- `g1.h100` — NVIDIA H100 (80GB)
- `g1.rtx6000` — NVIDIA RTX 6000

The `CC-Ubuntu24.04-CUDA` image has CUDA drivers pre-installed for all of these.

In [ ]:
context.choose_site(default="KVM@TACC")

gpu_lease = chi.lease.get_lease(GPU_LEASE_NAME)
print(f"GPU lease: {gpu_lease.name} — status: {gpu_lease.status}")

reserved_flavors = gpu_lease.get_reserved_flavors()
print(f"Reserved GPU flavor: {[f.name for f in reserved_flavors]}")

if not reserved_flavors:
    raise RuntimeError(f"Lease {GPU_LEASE_NAME} has no reserved flavors. Check the lease in the Chameleon UI.")

gpu_node = server.Server(
    GPU_NODE_NAME,
    image_name=GPU_IMAGE_NAME,
    flavor_name=reserved_flavors[0].name,
)
gpu_node.submit(idempotent=True)
print(f"GPU node submitted with flavor: {reserved_flavors[0].name}")
print("Waiting for GPU node to become active...")

---
## Cell 10 — Configure GPU Node Network

In [ ]:
gpu_node.associate_floating_ip()
gpu_node.refresh()
gpu_node.check_connectivity()

for sg_name in ["allow-ssh", "allow-8888"]:
    gpu_node.add_security_group(sg_name)

gpu_ip = gpu_node.get_floating_ip()
print(f"GPU node is up: {gpu_ip}")
print(f"GPU flavor: {reserved_flavors[0].name}")

---
## Cell 11 — Install Docker + NVIDIA Toolkit on GPU Node

Installs Docker and the NVIDIA container toolkit so the training container
can access the GPU. The `CC-Ubuntu24.04-CUDA` image already has CUDA drivers —
this just adds the container runtime layer.

In [ ]:
gpu_node.execute("curl -sSL https://get.docker.com/ | sudo sh")
gpu_node.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
gpu_node.execute("""
curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey \
  | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg
curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list \
  | sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' \
  | sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list
sudo apt-get update
sudo apt-get install -y nvidia-container-toolkit
sudo nvidia-ctk runtime configure --runtime=docker
sudo systemctl restart docker
""")

# Verify GPU is visible
result = gpu_node.execute("nvidia-smi 2>&1")
print(result)
print("Docker + NVIDIA toolkit installed")

---
## Cell 12 — Clone Repo + Build Training Docker Image

Clones the public GitHub repo and builds the training container image.
No GitHub token needed — the repo is public.

The Docker image contains:
- PyTorch + CUDA
- HuggingFace transformers, PEFT, datasets
- MLflow client
- boto3 (for S3/Swift access)
- All training code from `training/train.py`

In [ ]:
gpu_node.execute(f"git clone https://github.com/{GITHUB_REPO}.git ~/proj30 2>&1")

print("Building Docker image (this takes 5-10 minutes)...")
result = gpu_node.execute(
    "cd ~/proj30/training && docker build -t proj30-train:latest . 2>&1"
)
print(result)
print("Training image built successfully")

---
## Cell 13 — Write Credentials File + Clone Repo on MLflow VM

Writes `/home/cc/.mlflow_s3_credentials` on the MLflow VM. This file is
sourced by the cron job before running `retrain_trigger.py`.

See the introduction at the top of this notebook for a description of each variable.

In [ ]:
context.choose_site(default="KVM@TACC")

# Clone repo onto MLflow VM (needed for retrain_trigger.py and rollback.py)
s.execute(f"git clone https://github.com/{GITHUB_REPO}.git ~/proj30 2>&1")

# Write credentials file — overwrites any previous version cleanly
s.execute(f"""cat > /home/cc/.mlflow_s3_credentials << 'CREDEOF'
# Chameleon EC2-compatible credentials for object store access
# Auto-generated by setup notebook — do not commit to git
AWS_ACCESS_KEY_ID={AWS_ACCESS_KEY_ID}
AWS_SECRET_ACCESS_KEY={AWS_SECRET_ACCESS_KEY}

# MLflow server location
MLFLOW_TRACKING_URI=http://{mlflow_ip}:8000
MLFLOW_S3_ENDPOINT_URL=https://chi.tacc.chameleoncloud.org:7480

# Buckets
MLFLOW_ARTIFACTS_BUCKET={MLFLOW_ARTIFACTS_BUCKET}
TRAINING_DATA_BUCKET={TRAINING_DATA_BUCKET}

# GitHub repo (public — no token needed)
GITHUB_REPO={GITHUB_REPO}

# Chameleon project
OS_PROJECT_NAME=CHI-251409

# GPU node SSH target for cron trigger
GPU_NODE_IP={gpu_ip}

# Inference server reload endpoint (set when teammate deploys inference server)
INFERENCE_SERVER_URL={INFERENCE_SERVER_URL}
CREDEOF
""")

# Verify
result = s.execute("cat /home/cc/.mlflow_s3_credentials")
print(result)
print("Credentials file written to /home/cc/.mlflow_s3_credentials")

---
## Cell 14 — Upload SSH Key to MLflow VM

The cron trigger SSHes from the MLflow VM into the GPU node to run training.
This cell copies your Trovi SSH key to the MLflow VM so that SSH works
without a password.

In [ ]:
ssh_key_path = os.path.expanduser(SSH_KEY_PATH)

if not os.path.exists(ssh_key_path):
    # Try to find the key automatically
    for candidate in ["~/work/.ssh/id_rsa", "~/.ssh/id_rsa", "~/.ssh/id_ed25519"]:
        expanded = os.path.expanduser(candidate)
        if os.path.exists(expanded):
            ssh_key_path = expanded
            print(f"Found SSH key at: {ssh_key_path}")
            break
    else:
        raise FileNotFoundError(
            f"SSH key not found at {SSH_KEY_PATH}. "
            f"Update SSH_KEY_PATH in Cell 1 to point to your key."
        )

s.execute("mkdir -p ~/.ssh && chmod 700 ~/.ssh")
s.upload(ssh_key_path, "/home/cc/.ssh/id_rsa_chameleon")
s.execute("chmod 600 ~/.ssh/id_rsa_chameleon")
print(f"SSH key uploaded from {ssh_key_path}")

# Verify
result = s.execute("ls -la ~/.ssh/")
print(result)

---
## Cell 15 — Set Up Weekly Cron Job

Installs a cron job on the MLflow VM that runs `retrain_trigger.py` every
Sunday at 2am. The trigger SSHes into the GPU node and runs the training
container. Logs are written to `/home/cc/retrain.log`.

In [ ]:
cron_line = (
    "0 2 * * 0 "
    "set -a; source /home/cc/.mlflow_s3_credentials; set +a; "
    "python3 /home/cc/proj30/training/retrain_trigger.py "
    ">> /home/cc/retrain.log 2>&1"
)

# Add cron job (removes any existing retrain_trigger line first to avoid duplicates)
s.execute(f'(crontab -l 2>/dev/null | grep -v retrain_trigger; echo "{cron_line}") | crontab -')

result = s.execute("crontab -l")
print(f"Cron jobs on MLflow VM:\n{result}")
print("\nRetraining will run automatically every Sunday at 2am.")
print("Logs: /home/cc/retrain.log on the MLflow VM")

---
## Cell 16 — Smoke Test

Runs a single training epoch end to end to verify the full pipeline works:
- Data loads from the object store bucket
- MeetingBank adapter downloads from MLflow
- Training runs on the GPU
- Metrics are logged to MLflow

Gate and model registration are skipped in smoke test mode.
Expected output: `[done] exit code: 0`

**Expected runtime: ~5 minutes**

In [ ]:
print("Running smoke test (1 epoch, skips gate+registration)...\n")

result = gpu_node.execute(f"""
    docker run --rm --gpus all \\
        -e MLFLOW_TRACKING_URI=http://{mlflow_ip}:8000 \\
        -e MLFLOW_S3_ENDPOINT_URL=https://chi.tacc.chameleoncloud.org:7480 \\
        -e AWS_ACCESS_KEY_ID={AWS_ACCESS_KEY_ID} \\
        -e AWS_SECRET_ACCESS_KEY={AWS_SECRET_ACCESS_KEY} \\
        -e BUCKET_NAME={TRAINING_DATA_BUCKET} \\
        -e DATA_PREFIX=raw/ \\
        -v /mnt/data:/mnt/data \\
        proj30-train:latest --smoke-test 2>&1
""")
print(result)

print("\n" + "="*60)
print("SETUP COMPLETE")
print("="*60)
print(f"MLflow UI:        http://{mlflow_ip}:8000")
print(f"GPU node IP:      {gpu_ip}")
print(f"Cron schedule:    Every Sunday at 2am")
print(f"Retrain logs:     ssh cc@{mlflow_ip} 'tail -f /home/cc/retrain.log'")
print(f"\nTo run a full training run now, proceed to Cell 17.")

---
## Cell 17 — Full Training Run (optional)

Runs a full 3-epoch training run. On success:
- Model is registered in MLflow as `bart-meeting-summarizer`
- If rougeL beats current staging: alias `staging` is updated
- If rougeL beats current production: alias `production` is updated automatically

**Expected runtime: ~20-30 minutes depending on GPU**

In [ ]:
print("Running full training (3 epochs)...\n")

result = gpu_node.execute(f"""
    docker run --rm --gpus all \\
        -e MLFLOW_TRACKING_URI=http://{mlflow_ip}:8000 \\
        -e MLFLOW_S3_ENDPOINT_URL=https://chi.tacc.chameleoncloud.org:7480 \\
        -e AWS_ACCESS_KEY_ID={AWS_ACCESS_KEY_ID} \\
        -e AWS_SECRET_ACCESS_KEY={AWS_SECRET_ACCESS_KEY} \\
        -e BUCKET_NAME={TRAINING_DATA_BUCKET} \\
        -e DATA_PREFIX=raw/ \\
        -e INFERENCE_SERVER_URL={INFERENCE_SERVER_URL} \\
        -v /mnt/data:/mnt/data \\
        proj30-train:latest 2>&1
""")
print(result)

---
## Cell 18 — Test Automated Trigger (optional)

Manually fires `retrain_trigger.py` from the MLflow VM — the same script
the cron job runs automatically on Sundays. Use this to verify the
end-to-end automation works without waiting for Sunday.

In [ ]:
print("Manually triggering retrain_trigger.py on MLflow VM...\n")

result = s.execute("""
    set -a
    source /home/cc/.mlflow_s3_credentials
    set +a
    python3 /home/cc/proj30/training/retrain_trigger.py
""")
print(result)

---
## Cell 19 — Reconnect to Existing VMs

Run this cell if you restart the notebook kernel and need to reconnect
to already-running VMs without reprovisioning them.

In [ ]:
context.choose_project()
context.choose_site(default="KVM@TACC")

s = server.Server(SERVER_NAME)
s.refresh()
mlflow_ip = s.get_floating_ip()
print(f"MLflow VM: {s.status} — {mlflow_ip}")

gpu_node = server.Server(GPU_NODE_NAME)
gpu_node.refresh()
gpu_ip = gpu_node.get_floating_ip()
print(f"GPU node:  {gpu_node.status} — {gpu_ip}")

# Reload credentials
env_file = os.path.expanduser("~/work/.mlflow_s3_credentials")
credentials = {}
with open(env_file) as f:
    for line in f:
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, value = line.split("=", 1)
            credentials[key] = value
AWS_ACCESS_KEY_ID     = credentials["AWS_ACCESS_KEY_ID"]
AWS_SECRET_ACCESS_KEY = credentials["AWS_SECRET_ACCESS_KEY"]
print("Credentials reloaded")
print(f"\nMLflow UI: http://{mlflow_ip}:8000")

---
## Cell 20 — Cleanup (only run when done with the project)

**!!Warning!!** This deletes the MLflow VM and detaches the volume.
MLflow data is preserved in the persistent volume and can be reattached
if you relaunch the VM. The GPU node lease will expire on its own.

Only run this when you are completely done with the project.

In [ ]:
# Uncomment to run cleanup

# # Unmount and detach volume (preserves PostgreSQL data)
# s.execute("sudo umount /mnt/mlflow_persist")
# volume = cinder_client.volumes.get(volume.id)
# s.detach_volume(volume.id)
# print("Volume detached — data preserved")

# # Delete MLflow VM
# s.delete()
# print("MLflow VM deleted")

print("Cleanup cell — uncomment lines above to run")